# 1a · Explore & clean the colony telemetry

**Core · Live-code · ~25 min**

In this notebook we do the very first thing any data or AI professional does with a new
dataset: **open it up and look at it.** We'll load the colony's sensor readings, check them
for problems, fix a few gaps, and draw some charts so we can literally *see* the incidents
hiding in the week.

**How this notebook works:** each code cell has a `# TODO` for you to complete. Read the
explanation above each cell first — it tells you exactly what to write and why. If you get
stuck, the finished version is in [`solution/01a_explore_clean.ipynb`](solution/01a_explore_clean.ipynb).

To run a cell: click it and press **Shift+Enter**.

## Step 1 — Load the data

We use **pandas**, the standard Python library for working with tables of data. Pandas gives
us a **DataFrame**: essentially a programmable spreadsheet. You can think of `df` (our
DataFrame) as the whole telemetry table sitting in memory, ready to be sliced, filtered, and
plotted.

- `pd.read_csv(...)` reads our comma-separated file into a DataFrame.
- `parse_dates=["timestamp"]` tells pandas that the `timestamp` column is a *date/time*, not
  just text — that's what lets us plot things over time later.
- `df.head()` shows the first 5 rows so we can eyeball what we loaded.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# read_csv loads the file into a DataFrame (a table). The path is relative to this notebook.
df = pd.read_csv("../data/telemetry.csv", parse_dates=["timestamp"])

print("rows, columns:", df.shape)   # (number of rows, number of columns)
df.head()                            # show the first few rows

## Step 2 — Inspect it

Before trusting any data, we ask three questions:
1. **What columns are there and what type is each?** (`df.info()`)
2. **What do the numbers look like** — their averages, minimums, maximums? (`df.describe()`)
3. **Are any values missing?** (`df.isna().sum()`)

`df.info()` prints each column, how many non-missing values it has, and its data type.
`df.describe()` gives summary statistics for the numeric columns — a fast way to sanity-check
that, say, oxygen is hovering around 21% and not 2100%.

In [ ]:
# TODO: print a structural summary of the DataFrame.
# Write:  df.info()

# TODO: on the next line, show summary statistics for every numeric column.
# Write:  df.describe()

### Are there missing values?

Real sensors drop readings — a glitch, a reboot, a lost packet. In a CSV those show up as
empty cells, which pandas represents as `NaN` ("Not a Number"). We deliberately removed a few
readings so you can practise handling this.

`df.isna()` returns a table of True/False (True = missing). Summing it per column
(`.sum()`) counts how many are missing in each column.

In [ ]:
# TODO: count the missing values in each column.
# Write:  df.isna().sum()

## Step 3 — Clean the missing readings

We have to decide what to do about the gaps. Deleting whole rows would throw away good data
in the *other* columns. A better choice for a smoothly-changing time series is to **carry the
last known value forward** — if oxygen was 21.0% and the next reading is missing, 21.0% is a
very reasonable stand-in for 15 minutes later.

- `df.ffill()` = *forward fill*: replace each gap with the value **before** it.
- `df.bfill()` = *back fill*: replace any remaining gaps (e.g. a gap on the very first row,
  which has nothing before it) with the value **after** it.

Chaining `.ffill().bfill()` covers every case.

In [ ]:
# TODO: fill the gaps by forward-filling then back-filling, and store the result back in df.
# Write:  df = df.ffill().bfill()

# TODO: confirm there are zero missing values now.
# Write:  df.isna().sum()

## Step 4 — Plot the week and find the incidents

Numbers in a table are hard for humans to interpret; **charts** are easy. We'll plot each
signal against time. A healthy signal wiggles gently around its normal range; an incident
looks like a sudden spike or dip that breaks the pattern.

`plt.plot(x, y)` draws a line. We give it the timestamp on the x-axis and a signal on the
y-axis. `plt.title(...)` labels it, and `plt.show()` renders it.

**What to look for:** somewhere around Day 5, oxygen dips *while* CO₂ climbs — that's the
scrubber fault. See if you can spot it.

In [ ]:
# TODO: plot oxygen over time.
# Write these three lines:
#   plt.plot(df["timestamp"], df["o2_pct"])
#   plt.title("Oxygen % over the week")
#   plt.show()

In [ ]:
# TODO: now plot co2_ppm and power_kw the same way (one chart each is fine).
#
# Question to answer for yourself: on which day do you see O2 fall AND CO2 rise together?
# That combination is the tell-tale sign of a CO2 scrubber problem.

### 🌱 Stretch (optional)

- **Highlight the known incidents.** Because we have the `label` column, you can colour the
  incident points red on your oxygen chart to confirm your eye was right. See the solution
  for how.
- **Which signals move together?** `df[["o2_pct", "co2_ppm", "power_kw"]].corr()` computes
  *correlation* — a number from -1 to 1 describing whether two signals rise/fall together.
  You'd expect O₂ and CO₂ to be negatively correlated during a scrubber fault.

## ✅ Recap

You loaded real-looking sensor data, inspected it, cleaned missing values, and used charts to
*see* problems that would be invisible in a table. This "look before you model" habit is the
foundation everything else today is built on. Next (**1b**) we'll get a computer to spot
those incidents automatically.

# 1a · Explore & clean the colony telemetry

**Core · Live-code · ~25 min**

Goal: load the Orbital telemetry, clean a few missing readings, and plot the week so we can *see* the incidents.

> Fill in each `# TODO`. Stuck? Peek at [`solution/01a_explore_clean.ipynb`](solution/01a_explore_clean.ipynb).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# The dataset is created by data/generate_telemetry.py during setup.
df = pd.read_csv("../data/telemetry.csv", parse_dates=["timestamp"])
print(df.shape)
df.head()

## Inspect the data

How many rows? What are the column types? Are there missing values?

In [ ]:
# TODO: print df.info() to see column types and non-null counts

# TODO: use df.describe() to see basic statistics for each signal

In [ ]:
# TODO: how many missing values are in each column?
# hint: df.isna().sum()

## Clean the missing readings

A few sensors dropped readings. For time-series like this, filling a gap with the
previous value (forward fill) is a reasonable first step.

In [ ]:
# TODO: forward-fill missing values, then back-fill any leftovers at the very start
# hint: df = df.ffill().bfill()

# TODO: confirm there are no missing values left

## Plot the week

Plot oxygen, CO₂, and power over time. Can you spot the incidents by eye?

In [ ]:
# TODO: plot o2_pct against timestamp
# hint: plt.plot(df["timestamp"], df["o2_pct"]) ; plt.title(...) ; plt.show()

In [ ]:
# TODO: plot co2_ppm and power_kw too (separate charts or subplots)
# Question: around which day do you see O2 fall while CO2 rises?

### 🌱 Stretch

- Highlight the labeled incidents on your O₂ chart (colour points where `label == 1`).
- Which signals move *together*? Try `df[[...]].corr()`.